In [3]:
# importing packages
from pytubefix import YouTube
import os

# url input from youtube
yt = YouTube("https://youtu.be/Q9Hmo3EsvXA?si=9OQ3OARFLFSTtFNN")

# extract only audio
video = yt.streams.filter(only_audio=True).first()

# set destination to save file
destination = ("C:/Users/fireh/Documents/GitHub/2024-25c-fai2-adsai-group-arb-6/tasks/task-2/whisper")

# download the file
out_file = video.download(output_path=destination)

# save the file
base, ext = os.path.splitext(out_file)
new_file = base + '.mp3'
os.rename(out_file, new_file)

In [1]:
import os
os.listdir()

['blind_date.mp3', 'speech_to_text_whisper1.ipynb', '__pycache__']

In [ ]:
import whisper
import pandas as pd

# Load Whisper model (choose "medium", "small", or "large" for better accuracy)
model = whisper.load_model("large-v2", device="cpu")

# Path to your local audio file
file_path = r"blind_date.mp3"

# Transcribe the audio file with Arabic language support
transcription = model.transcribe(file_path, language="ar")

# Extract transcribed text
transcribed_text = transcription["text"]

# Save transcription to an Excel file
df = pd.DataFrame({"Transcription": [transcribed_text]})  # Create DataFrame
excel_path = r"C:/Users/fireh/Documents/GitHub/2024-25c-fai2-adsai-group-arb-6/tasks/task-2/whisper/transcribed_data_whisper1.xlsx"  # Save path
df.to_excel(excel_path, index=False, engine="openpyxl")  # Save as Excel

print(f"Transcription saved to: {excel_path}")


c:\Users\fireh\anaconda3\envs\block_b\lib\site-packages\whisper\transcribe.py:130: UserWarning: Performing inference on CPU when CUDA is available
  warnings.warn("Performing inference on CPU when CUDA is available")
c:\Users\fireh\anaconda3\envs\block_b\lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to: C:/Users/fireh/Documents/GitHub/2024-25c-fai2-adsai-group-arb-6/tasks/task-2/whisper/transcribed_data_whisper.xlsx


In [5]:
import os
import re
import pandas as pd
from deepmultilingualpunctuation import PunctuationModel

# ✅ Set file paths
EXCEL_PATH = r"C:/Users/fireh/Documents/GitHub/2024-25c-fai2-adsai-group-arb-6/tasks/task-2/whisper/transcribed_data_whisper.xlsx"
OUTPUT_CSV = r"C:/Users/fireh/Documents/GitHub/2024-25c-fai2-adsai-group-arb-6/tasks/task-2/whisper/transcribed_data_whisper1.csv"

# Punctuation restoration
def restore_punctuation(text):
    model = PunctuationModel()
    return model.restore_punctuation(text)

# Text cleaning functions
def split_text_into_sentences(text, max_word_count=20):
    sentences = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            parts = re.split(r'(?<=[\.!\؟؛])\s+', line)
            sentences.extend([part.strip() for part in parts if part.strip()])

    refined_sentences = []
    for sentence in sentences:
        words = sentence.split()
        if len(words) > max_word_count:
            subparts = re.split(r'(?<=[,،])\s+', sentence)
            for sp in subparts:
                sp = sp.strip()
                if sp:
                    sub_words = sp.split()
                    if len(sub_words) > max_word_count:
                        for i in range(0, len(sub_words), max_word_count):
                            refined_sentences.append(" ".join(sub_words[i:i+max_word_count]))
                    else:
                        refined_sentences.append(sp)
        else:
            refined_sentences.append(sentence)
    return refined_sentences

def merge_short_fragments(sentences, min_word_count=3):
    merge_starters = {'و', 'زواج', 'لكن', 'بس', 'كبير', 'كتير'}
    merged = []
    for s in sentences:
        s = s.strip()
        if merged:
            words = s.split()
            if len(words) < min_word_count or (words and words[0] in merge_starters):
                merged[-1] = merged[-1] + " " + s
                continue
        merged.append(s)
    return merged

def remove_english_artifacts(text):
    cleaned = re.sub(r'\b[A-Za-z]+\b', '', text)
    cleaned = re.sub(r'\s{2,}', ' ', cleaned)
    return cleaned.strip()

def clean_sentence(sentence):
    sentence = re.sub(r'^[,،\s]+', '', sentence)
    sentence = re.sub(r'[,،\s]+$', '', sentence)
    return sentence.strip()

def save_sentences_to_csv(sentences, file_path):
    df = pd.DataFrame(sentences, columns=["Sentence"])
    df.to_csv(file_path, index=False, encoding='utf-8-sig')
    print(f"✅ Processed text saved to: {file_path}")

# Run the pipeline
def run_processing():
    # Load transcribed text from Excel
    if not os.path.isfile(EXCEL_PATH):
        print("❌ Excel file not found:", EXCEL_PATH)
        return

    df = pd.read_excel(EXCEL_PATH, engine="openpyxl")
    if "Transcription" not in df.columns:
        print("❌ Column 'Transcription' not found in the Excel file")
        return
    
    # Assuming the transcription is in the first row
    transcript_text = df["Transcription"].iloc[0]

    print("✍ Restoring punctuation...")
    punctuated_text = restore_punctuation(transcript_text)

    print("🔍 Removing English artifacts...")
    cleaned_text = remove_english_artifacts(punctuated_text)

    print("📌 Splitting into sentences...")
    sentences = split_text_into_sentences(cleaned_text)
    sentences = merge_short_fragments(sentences)

    sentences = [clean_sentence(s) for s in sentences if s]

    print("💾 Saving to CSV...")
    save_sentences_to_csv(sentences, OUTPUT_CSV)

# Run the function
run_processing()


✍ Restoring punctuation...


config.json: 100%|██████████| 892/892 [00:00<00:00, 197kB/s]
model.safetensors: 100%|██████████| 2.24G/2.24G [04:01<00:00, 9.26MB/s]
tokenizer_config.json: 100%|██████████| 406/406 [00:00<00:00, 89.4kB/s]
sentencepiece.bpe.model: 100%|██████████| 5.07M/5.07M [00:02<00:00, 2.19MB/s]
tokenizer.json: 100%|██████████| 17.1M/17.1M [00:02<00:00, 5.71MB/s]
special_tokens_map.json: 100%|██████████| 239/239 [00:00<00:00, 65.5kB/s]
c:\Users\fireh\anaconda3\envs\block_b\lib\site-packages\transformers\pipelines\token_classification.py:169: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="none"` instead.
  warnings.warn(
c:\Users\fireh\anaconda3\envs\block_b\lib\site-packages\transformers\pipelines\base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


🔍 Removing English artifacts...
📌 Splitting into sentences...
💾 Saving to CSV...
✅ Processed text saved to: C:/Users/fireh/Documents/GitHub/2024-25c-fai2-adsai-group-arb-6/tasks/task-2/whisper/transcribed_data_whisper1.csv
